# 🔬 ResNet-50 Transfer Learning — CIFAR-10

**Dataset:** CIFAR-10 — 60,000 RGB images, 10 classes  
**Task:** Transfer ImageNet knowledge to CIFAR-10 using two-stage fine-tuning

## Why Transfer Learning?

Training ResNet-50 from scratch on CIFAR-10 requires millions of iterations. ImageNet-pretrained ResNet-50 already learned powerful low-level features (edges, textures) and mid-level features (shapes, parts). Transfer learning reuses these features:

| Stage | What trains | Learning rate | Epochs |
|---|---|---|---|
| Feature extraction | fc layer only | 1e-3 | 5 |
| Fine-tuning | layer3 + layer4 + fc | 1e-4 | 10 |

## ResNet-50 Architecture

ResNet-50 uses residual connections to solve the vanishing gradient problem:

$$\mathcal{F}(x) = H(x) - x \implies H(x) = \mathcal{F}(x) + x$$

Instead of learning the full mapping $H(x)$, the network learns the residual $\mathcal{F}(x)$. If the optimal mapping is close to identity, $\mathcal{F}(x) \approx 0$ is easier to learn than $H(x) \approx x$.

**Bottleneck block (used in ResNet-50):**
```
x → Conv(1×1) → BN → ReLU          ← reduce channels
  → Conv(3×3) → BN → ReLU          ← spatial processing
  → Conv(1×1) → BN                  ← restore channels
  → + x (skip connection) → ReLU
```

## Why unfreeze layer3 + layer4 specifically?

ResNet-50 layers learn features at different abstraction levels:
- **layer1, layer2** — low-level: edges, textures, colors (universal — keep frozen)
- **layer3** — mid-level: object parts, patterns (fine-tune for new domain)
- **layer4** — high-level: class-specific features (most important to fine-tune)
- **fc** — classification head (always retrain for new classes)

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import yaml
import torch
import matplotlib.pyplot as plt

from src.resnet.model   import ResNet50Transfer
from src.resnet.trainer import ResNetTrainer

with open('../configs/resnet_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Config:', config)

In [ ]:
# Model summary — verify trainable parameters per stage
model = ResNet50Transfer(
    num_classes = config['model']['num_classes'],
    dropout     = config['model']['dropout'],
).to(device)

print('Stage 1 — Feature Extraction (fc only):')
print(f'  Trainable params: {model.count_parameters():,}')

model.unfreeze_layers(['layer3', 'layer4'])
print('\nStage 2 — Fine-Tuning (layer3 + layer4 + fc):')
print(f'  Trainable params: {model.count_parameters():,}')

# reset to stage 1
model.freeze_backbone()

# verify forward pass
x_test = torch.rand(2, 3, 224, 224).to(device)
out    = model(x_test)
print(f'\nInput  : {x_test.shape}')
print(f'Output : {out.shape}  ← should be (2, 10)')

In [ ]:
# Train — two stage training happens automatically
# CPU: Stage 1 ~15 mins, Stage 2 ~30 mins (ResNet-50 on 224x224 is slow on CPU)
# GPU: ~5-8 mins total
trainer = ResNetTrainer(config)
trainer.train()

history = trainer.get_history()
print(f'\nFinal Train Acc : {history["train_accs"][-1]*100:.2f}%')
print(f'Final Val Acc   : {history["val_accs"][-1]*100:.2f}%')

In [ ]:
# Plot training curves with stage boundary marked
history  = trainer.get_history()
epochs   = range(1, len(history['train_losses']) + 1)
boundary = history['stage_boundary']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, train_data, val_data, title in zip(
    [ax1, ax2],
    [history['train_losses'], [a*100 for a in history['train_accs']]],
    [history['val_losses'],   [a*100 for a in history['val_accs']]],
    ['Loss', 'Accuracy (%)'],
):
    ax.plot(epochs, train_data, label='Train', color='#3498DB')
    ax.plot(epochs, val_data,   label='Val',   color='#E74C3C')
    ax.axvline(x=boundary, color='green', linestyle='--',
               alpha=0.8, label='Fine-tuning starts')
    ax.set_title(f'ResNet-50 — {title}')
    ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('ResNet-50 Transfer Learning — CIFAR-10', fontsize=13)
plt.tight_layout()
plt.show()
print('Note: green dashed line = where fine-tuning (layer3+layer4) begins')

In [ ]:
# Evaluate on test set
acc, cm = trainer.evaluate()
print(f'\nTest Accuracy: {acc*100:.2f}%')

classes    = trainer.classes
per_class  = cm.diagonal() / cm.sum(axis=1)
print('\nPer-class accuracy:')
for cls, acc_c in zip(classes, per_class):
    print(f'  {cls:<12}: {acc_c*100:.1f}%')

In [ ]:
# CNN vs ResNet comparison — fill after training both
results = {
    'Model'      : ['Custom CNN', 'ResNet-50 Transfer'],
    'Parameters' : ['~1.2M',      '~25M (23M frozen)'],
    'Test Acc'   : ['fill',        'fill'],
    'Epochs'     : [15,             15],
    'Train Time' : ['fill',         'fill'],
}

print('\nModel Comparison:')
print(f'{"Model":<25} {"Params":<22} {"Test Acc":<12} {"Epochs"}')
print('-' * 65)
for i in range(len(results['Model'])):
    print(
        f'{results["Model"][i]:<25} '
        f'{results["Parameters"][i]:<22} '
        f'{results["Test Acc"][i]:<12} '
        f'{results["Epochs"][i]}'
    )

## Results

| Model | Parameters | Test Accuracy | Epochs |
|---|---|---|---|
| Custom CNN | ~1.2M | *fill* | 15 |
| ResNet-50 Transfer | ~25M (23M frozen S1) | *fill* | 15 (5+10) |

## Key Takeaways

**Feature extraction first, then fine-tune** — jumping straight to fine-tuning with a high learning rate destroys pretrained weights. Always warm up the fc layer first.

**Lower lr for fine-tuning** — pretrained weights are already good. A 10× smaller lr nudges them toward the new domain without catastrophic forgetting.

**ResNet-50 on 32×32 is suboptimal** — the model was designed for 224×224. Resizing CIFAR-10 images to 224×224 works but loses the native low-resolution texture that the CNN from scratch learns to exploit directly.